# DESC 3D continuum with STELLGAP

DESC hdf5 + NTP.mat -> VMEC -> BOOZ_XFORM -> STELLGAP.

In [ ]:
################## 1. 用户输入区 ##################

# 文件与剖面
EQUILIBRIUM_HDF5 = r'C:\Users\Desktop\equil.hdf5'  # DESC hdf5 平衡；填当前 case。
PROFILE_MAT = r'C:\Users\Desktop\NTP.mat'           # NTP.mat；需含 rhoSample/neSample/niSample/TeSample。

# 剖面拟合
NE_POLY_DEGREE = 0            # ne(rho) 多项式次数；仅用于检查图，0~9。
NI_POLY_DEGREE = 0            # ni(rho) 多项式次数；0~9，受 STELLGAP 输入限制。
TE_POLY_DEGREE = 0            # Te(rho) 多项式次数；0~9，声波耦合时使用。

# 物理模型
ION_MASS_IN_PROTONS = 1.0           # 背景离子质量 mi/mp；H=1，D=2。
SOUND_COUPLING = True               # True: xstgap_snd_ver7；False: xstgap。
SOUND_GAMMA = 1.35                  # 声波绝热指数；常用 1.35。
SLOW_SOUND_APPROX = True            # True: 慢声近似；False: 保留密集声波分支。

# 模数
GAP_TYPE = 'MAE'                    # 'TAE'、'MAE' 或 'HAE'；影响旁带扩展策略。
MODE_PAIR_NM = [(6, 14), (10, 14)]  # 物理标记 (n, m)；TAE 例子 [(6, 10), (6, 11)]。
RUN_LEVELS = ('large',)             # 旁带截断级别；可选 bare/compact/medium/large。

# 分辨率
VMEC_SURFACES = 128                 # DESC->VMEC 径向面数；建议 101~201。
VMEC_M_NYQ, VMEC_N_NYQ = 18, 10     # VMEC Nyquist 截断；需覆盖平衡和目标旁带。
VMEC_M_GRID, VMEC_N_GRID = 72, 40   # DESC->VMEC 角向网格；建议 >= 4*Nyquist。
BOOZER_M, BOOZER_N = 36, 24         # BOOZ_XFORM 截断；需覆盖目标模和旁带。
METRIC_THETA, METRIC_ZETA = 72, 48 # STELLGAP 度规角向网格；建议 >= 2*Boozer 截断。
PROFILE_POINTS = 128                # q/iota 剖面采样；建议 201。
RADIAL_POINTS = 128                # STELLGAP 径向扫描点数；建议 501~1001 做最终图。

# 画图
PLOT_RHO_RANGE = (0.10, 0.90)        # 显示的 rho=sqrt(s) 范围；不改变计算。
FREQUENCY_RANGE_KHZ =  (200, 600.0)          # 纵坐标范围；None 表示自动。
REFERENCE_FREQUENCY_KHZ = None      # 参考频率横线；不用则设 None。

In [ ]:
################## 2. 读取并拟合 NTP ##################

from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat


def to_wsl_path(path):
    text = str(path).strip().strip('"').strip("'").replace(chr(92), '/')
    match = re.match(r'^([A-Za-z]):/(.*)$', text)
    return Path(f'/mnt/{match.group(1).lower()}/{match.group(2)}') if match else Path(text).expanduser()


def read_profile_mat(path):
    names = ['rhoSample', 'neSample', 'niSample', 'TeSample']
    try:
        mat = loadmat(path)
        return tuple(np.ravel(mat[name]).astype(float) for name in names)
    except NotImplementedError:
        import h5py
        with h5py.File(path, 'r') as mat:
            return tuple(np.ravel(np.array(mat[name])).astype(float) for name in names)


def fit_polynomial(x, y, degree, name):
    degree = int(degree)
    if not (0 <= degree <= 9):
        raise ValueError(f'{name} polynomial degree must be between 0 and 9.')
    if degree >= len(x):
        raise ValueError(f'{name} polynomial degree must be smaller than sample count.')
    coeff = np.polynomial.polynomial.polyfit(x, y, degree)
    fit = np.polynomial.polynomial.polyval(x, coeff)
    return coeff, fit


profile_path = to_wsl_path(PROFILE_MAT)
if not profile_path.exists():
    raise FileNotFoundError(profile_path)

rho_profile, ne_sample, ni_sample, te_sample = read_profile_mat(profile_path)
idx = np.argsort(rho_profile)
rho_profile = rho_profile[idx]
ne_sample, ni_sample, te_sample = ne_sample[idx], ni_sample[idx], te_sample[idx]
if np.any(~np.isfinite(rho_profile)) or np.any(~np.isfinite(ne_sample)) or np.any(~np.isfinite(ni_sample)) or np.any(~np.isfinite(te_sample)):
    raise ValueError('profile mat contains non-finite values.')
if rho_profile[0] < -1e-12 or rho_profile[-1] > 1 + 1e-12:
    print('Warning: rhoSample is expected to be in [0, 1].')

ne_coeff_rho, ne_fit = fit_polynomial(rho_profile, ne_sample, NE_POLY_DEGREE, 'ne')
ni_coeff_rho, ni_fit = fit_polynomial(rho_profile, ni_sample, NI_POLY_DEGREE, 'ni')
te_coeff_rho, te_fit = fit_polynomial(rho_profile, te_sample, TE_POLY_DEGREE, 'Te')
ni_coeff_s, _ = fit_polynomial(rho_profile**2, ni_sample, NI_POLY_DEGREE, 'ni(s)')
ni0_norm = float(ni_coeff_rho[0])
ni0_norm_s = float(ni_coeff_s[0])
te0_kev = float(te_coeff_rho[0])
if ni0_norm <= 0 or ni0_norm_s <= 0:
    raise ValueError(f'fitted ni(0) must be positive, got rho-fit={ni0_norm:.6e}, s-fit={ni0_norm_s:.6e}.')
if te0_kev <= 0:
    raise ValueError(f'fitted Te(0) must be positive, got {te0_kev:.6e} keV.')
if np.min(ne_fit) <= 0 or np.min(ni_fit) <= 0 or np.min(te_fit) <= 0:
    print('Warning: fitted profile has non-positive values.')

PROFILE_FIT = dict(
    mat_file=str(profile_path),
    rho=rho_profile,
    ne_sample=ne_sample,
    ni_sample=ni_sample,
    te_sample=te_sample,
    ne_fit=ne_fit,
    ni_fit=ni_fit,
    te_fit=te_fit,
    ne_coeff_rho=ne_coeff_rho,
    ni_coeff_rho=ni_coeff_rho,
    te_coeff_rho=te_coeff_rho,
    nion_coeff_rho=ni_coeff_rho / ni0_norm,
    nion_coeff_s=ni_coeff_s / ni0_norm_s,
    telec_coeff_rho=te_coeff_rho / te0_kev,
    ni0_m3=ni0_norm * 1e19,
    ni0_m3_s=ni0_norm_s * 1e19,
    te0_ev=te0_kev * 1e3,
)

def format_y_axis(axis, values, scientific=False, min_scale=1.0):
    values = np.asarray(values, dtype=float)
    style = 'sci' if scientific else 'plain'
    axis.ticklabel_format(axis='y', style=style, useOffset=False)
    ymin, ymax = float(np.nanmin(values)), float(np.nanmax(values))
    if np.isfinite(ymin) and np.isfinite(ymax) and abs(ymax - ymin) < 1e-10 * max(abs(ymin), abs(ymax), float(min_scale)):
        pad = 0.05 * max(abs(ymin), abs(ymax), float(min_scale))
        axis.set_ylim(ymin - pad, ymax + pad)


eps = 1e-12
fig, ax = plt.subplots(2, 3, figsize=(15, 7.2), sharex=True)
profiles = [
    (ne_sample, ne_fit, int(NE_POLY_DEGREE), r'$n_e$ [$10^{19}m^{-3}$]', 'Electron density'),
    (ni_sample, ni_fit, int(NI_POLY_DEGREE), r'$n_i$ [$10^{19}m^{-3}$]', 'Ion density'),
    (te_sample, te_fit, int(TE_POLY_DEGREE), r'$T_e$ [keV]', 'Electron temperature'),
]
for j, (sample, fit, degree, ylabel, title) in enumerate(profiles):
    ax[0, j].plot(rho_profile, sample, 'o', ms=4, label='sample')
    ax[0, j].plot(rho_profile, fit, '-', lw=2, label=f'fit deg={degree}')
    ax[0, j].set(ylabel=ylabel, title=title)
    format_y_axis(ax[0, j], np.r_[sample, fit])
    relative_error = 100 * (fit - sample) / np.maximum(np.abs(sample), eps)
    ax[1, j].plot(rho_profile, relative_error, lw=1.5)
    ax[1, j].set(xlabel=r'$\rho=\sqrt{s}$', ylabel='relative error [%]')
    format_y_axis(ax[1, j], relative_error, scientific=True, min_scale=1e-12)
for item in ax.ravel():
    item.grid(alpha=0.25)
for item in ax[0, :]:
    item.legend(fontsize=9)
fig.suptitle('Profile fit in rho', fontsize=15)
fig.tight_layout()
plt.show()

print(f'Using ni(0) = {PROFILE_FIT["ni0_m3"]:.6e} m^-3')
print(f'Using Te(0) = {PROFILE_FIT["te0_ev"]:.6e} eV')
if SOUND_COUPLING:
    print('nion coefficients for STELLGAP ion_profile=4:')
    print(np.array2string(PROFILE_FIT['nion_coeff_rho'], precision=8, separator=', '))
    print('telec coefficients for STELLGAP te_profile=2:')
    print(np.array2string(PROFILE_FIT['telec_coeff_rho'], precision=8, separator=', '))
else:
    print('nion coefficients for STELLGAP ion_profile=1:')
    print(np.array2string(PROFILE_FIT['nion_coeff_s'], precision=8, separator=', '))

In [ ]:
################## 3. 函数与工具 ##################

from pathlib import Path
from pprint import pprint
import os, re, shutil, subprocess, tempfile, time

# 限制 JAX 内存。
os.environ.setdefault('JAX_PLATFORM_NAME', 'cpu')
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.25')

import numpy as np
import matplotlib.pyplot as plt
from desc.io import load
from desc.grid import LinearGrid
from desc.vmec import VMECIO

TEMP_DIR = tempfile.TemporaryDirectory(prefix='continuum3D_')
WORK_DIR = Path(TEMP_DIR.name)

SOUND_COUPLING = bool(SOUND_COUPLING)
STELLGAP_PROGRAM = 'xstgap_snd_ver7' if SOUND_COUPLING else 'xstgap'
POST_PROGRAM = 'xpost_process_snd' if SOUND_COUPLING else 'xpost_process'
CONTINUUM_LABEL = 'Alfven-sound coupled' if SOUND_COUPLING else 'Alfven'


def to_wsl_path(path):
    text = str(path).strip().strip('"').strip("'").replace(chr(92), '/')
    match = re.match(r'^([A-Za-z]):/(.*)$', text)
    return Path(f'/mnt/{match.group(1).lower()}/{match.group(2)}') if match else Path(text).expanduser()


def load_equilibrium(path):
    obj = load(path)
    return obj if hasattr(obj, 'compute') else obj[-1]



def equilibrium_lmn_grid(eq):
    def as_int(value):
        if value is None:
            return None
        try:
            return int(value)
        except Exception:
            try:
                return int(np.asarray(value).reshape(-1)[0])
            except Exception:
                return None

    spectral = {name: as_int(getattr(eq, name, None)) for name in ['L', 'M', 'N']}
    grid = {name: as_int(getattr(eq, name, None)) for name in ['L_grid', 'M_grid', 'N_grid']}
    basis_max = {'L': 0, 'M': 0, 'N': 0}
    for basis_name in ['R_basis', 'Z_basis', 'L_basis']:
        basis = getattr(eq, basis_name, None)
        modes = getattr(basis, 'modes', None)
        if modes is None:
            continue
        modes = np.asarray(modes)
        if modes.ndim == 2 and modes.shape[1] >= 3 and modes.size:
            for key, col in [('L', 0), ('M', 1), ('N', 2)]:
                basis_max[key] = max(basis_max[key], int(np.max(np.abs(modes[:, col]))))
    for key in ['L', 'M', 'N']:
        spectral[key] = max(abs(spectral[key] or 0), basis_max[key])
    return {'spectral_LMN': spectral, 'grid_LMN': grid, 'basis_max_LMN': basis_max}


def check_resolution(eq, required_m, required_n=0, axisymmetric=False):
    res = equilibrium_lmn_grid(eq)
    eq_m = int(res['spectral_LMN']['M'] or 0)
    eq_n = 0 if axisymmetric else int(res['spectral_LMN']['N'] or 0)
    required_m = int(required_m)
    required_n = 0 if axisymmetric else int(required_n)
    warnings = []

    def warn_if(condition, message):
        if condition:
            warnings.append(message)

    vmec_m_nyq = int(VMEC_M_NYQ)
    vmec_m_grid = int(VMEC_M_GRID)
    boozer_m = int(BOOZER_M)
    metric_theta = int(METRIC_THETA)

    warn_if(vmec_m_nyq < max(eq_m, required_m),
            f'VMEC_M_NYQ={vmec_m_nyq} < required {max(eq_m, required_m)} (DESC M={eq_m}, continuum M={required_m}).')
    warn_if(vmec_m_grid < 4 * vmec_m_nyq,
            f'VMEC_M_GRID={vmec_m_grid} < recommended {4 * vmec_m_nyq} (=4*VMEC_M_NYQ).')
    warn_if(boozer_m < max(eq_m, 2 * required_m),
            f'BOOZER_M={boozer_m} < recommended {max(eq_m, 2 * required_m)} (>=DESC M and >=2*continuum M).')
    warn_if(metric_theta < 2 * boozer_m,
            f'METRIC_THETA={metric_theta} < recommended {2 * boozer_m} (=2*BOOZER_M).')

    if axisymmetric:
        warn_if(int(METRIC_ZETA) < 8, f'METRIC_ZETA={int(METRIC_ZETA)} is very low; use at least 8 for 2D.')
    else:
        vmec_n_nyq = int(VMEC_N_NYQ)
        vmec_n_grid = int(VMEC_N_GRID)
        boozer_n = int(BOOZER_N)
        metric_zeta = int(METRIC_ZETA)
        warn_if(vmec_n_nyq < eq_n,
                f'VMEC_N_NYQ={vmec_n_nyq} < DESC N={eq_n}.')
        warn_if(vmec_n_grid < 4 * vmec_n_nyq,
                f'VMEC_N_GRID={vmec_n_grid} < recommended {4 * vmec_n_nyq} (=4*VMEC_N_NYQ).')
        warn_if(boozer_n < max(eq_n, required_n),
                f'BOOZER_N={boozer_n} < required {max(eq_n, required_n)} (DESC N={eq_n}, continuum N≈{required_n}).')
        warn_if(metric_zeta < max(16, 2 * boozer_n),
                f'METRIC_ZETA={metric_zeta} < recommended {max(16, 2 * boozer_n)}.')

    print('DESC resolution:')
    pprint({'spectral_LMN': res['spectral_LMN'], 'grid_LMN': res['grid_LMN'],
            'required_MN': {'M': required_m, 'N': required_n}})
    if warnings:
        print('Resolution warnings:')
        for message in warnings:
            print('Warning:', message)
    else:
        print('Resolution check: OK.')
    return {'equilibrium': res, 'required_MN': {'M': required_m, 'N': required_n}, 'warnings': warnings}


def find_tool(name):
    path = shutil.which(name)
    if path:
        return path
    raise RuntimeError(f'Missing executable: {name}. Please activate conda env desc-0170.')


def run_command(cmd, log_name):
    print('RUN', ' '.join(map(str, cmd)), flush=True)
    start = time.time()
    with (WORK_DIR / log_name).open('w', encoding='utf-8') as stream:
        result = subprocess.run(cmd, cwd=WORK_DIR, stdout=stream, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f'Command failed ({result.returncode}); temporary log: {WORK_DIR / log_name}')
    print(f'DONE in {time.time() - start:.1f} s', flush=True)


def clear_stellgap_outputs():
    for pattern in ['alfven_spec*', 'all_alfven_spec', 'alfven_post', 'data_post', 'cond_no', 'gammas.dat']:
        for path in WORK_DIR.glob(pattern):
            if path.is_file() or path.is_symlink():
                path.unlink()


def fortran_bool(value):
    return 'T' if bool(value) else 'F'


def padded_coefficients(values, length=10):
    coeffs = list(map(float, values))
    if len(coeffs) > length:
        raise ValueError(f'Expected at most {length} coefficients, got {len(coeffs)}')
    return coeffs + [0.0] * (length - len(coeffs))


def normalize_run_levels(levels):
    if isinstance(levels, str):
        levels = (levels,)
    levels = tuple(str(level).lower().strip() for level in levels)
    allowed = {'bare', 'compact', 'medium', 'large'}
    bad = [level for level in levels if level not in allowed]
    if bad:
        raise ValueError(f'RUN_LEVELS contains unknown level(s): {bad}. Allowed: {sorted(allowed)}')
    if not levels:
        raise ValueError('RUN_LEVELS cannot be empty.')
    return levels


def write_plasma_input():
    profile_fit = globals().get('PROFILE_FIT')
    if profile_fit is None:
        raise RuntimeError('Run cell 2 to load NTP.mat before running STELLGAP.')
    # 普通 xstgap 不支持 ion_profile=4；仅声波版使用 rho profile。
    ion_density_0 = float(profile_fit['ni0_m3'] if SOUND_COUPLING else profile_fit['ni0_m3_s'])
    ion_profile = 4 if SOUND_COUPLING else 1
    nion_key = 'nion_coeff_rho' if SOUND_COUPLING else 'nion_coeff_s'
    lines = [
        '&plasma_input',
        f' ion_to_proton_mass={float(ION_MASS_IN_PROTONS):.10e},',
        f' ion_density_0={ion_density_0:.10e},',
        f' ion_profile={ion_profile},',
        ' jdqz_data=T,',
        ' egnout_form="asci",',
    ]
    nion = ', '.join(f'{x:.10e}' for x in padded_coefficients(profile_fit[nion_key]))
    lines.append(f' nion={nion},')
    if SOUND_COUPLING:
        telec = ', '.join(f'{x:.10e}' for x in padded_coefficients(profile_fit['telec_coeff_rho']))
        lines += [
            f' gamma={float(SOUND_GAMMA):.10e},',
            f' temp_elec_0={float(profile_fit["te0_ev"]):.10e},',
            ' te_profile=2,',
            f' telec={telec},',
            f' slow_sound={fortran_bool(SLOW_SOUND_APPROX)},',
        ]
    lines += ['/']
    (WORK_DIR / 'plasma.dat').write_text('\n'.join(lines) + '\n', encoding='ascii')


def read_stellgap_post(path):
    data = np.atleast_2d(np.loadtxt(path, skiprows=1))
    return data[np.isfinite(data).all(axis=1)]


def select_branch(data, m, n):
    branch = data[np.isclose(data[:, 2], m) & np.isclose(data[:, 3], n)]
    return branch[np.argsort(branch[:, 0])]


TOOLS = {
    'booz_xform': find_tool('xbooz_xform'),
    'metric': find_tool('xmetric'),
    'stgap': find_tool(STELLGAP_PROGRAM),
    'post': find_tool(POST_PROGRAM),
}
print(f'External tools ready: {STELLGAP_PROGRAM}, {POST_PROGRAM}')

In [ ]:
################## 4. 读平衡与目标面 ##################

eq_path = to_wsl_path(EQUILIBRIUM_HDF5)
if not eq_path.exists():
    raise FileNotFoundError(eq_path)

GAP_TYPE = str(GAP_TYPE).upper().strip()
RUN_LEVELS = normalize_run_levels(RUN_LEVELS)
if GAP_TYPE not in {'TAE', 'MAE', 'HAE'}:
    raise ValueError("GAP_TYPE must be 'TAE', 'MAE' or 'HAE'")
if len(MODE_PAIR_NM) != 2:
    raise ValueError('MODE_PAIR_NM must contain exactly two (n, m) pairs')
(n1, m1), (n2, m2) = [tuple(map(int, pair)) for pair in MODE_PAIR_NM]
if GAP_TYPE == 'TAE' and (n1 != n2 or abs(m1 - m2) != 1):
    print('Warning: a usual TAE pair has the same n and adjacent m.')
if GAP_TYPE == 'MAE' and m1 != m2:
    print('Warning: a usual MAE pair has the same m.')
if GAP_TYPE == 'HAE' and m1 == m2:
    print('Warning: a usual HAE pair has different m.')

eq = load_equilibrium(eq_path)

rho = np.linspace(0.0, 1.0, int(PROFILE_POINTS))
profile_grid = LinearGrid(rho=rho, M=0, N=0, NFP=eq.NFP)
iota = np.asarray(eq.compute(['iota'], grid=profile_grid)['iota']).reshape(-1)

# 目标交叉面。
iota_res = (n1 + n2) / (m1 + m2)
q_res = 1.0 / iota_res
if not np.nanmin(iota) <= iota_res <= np.nanmax(iota):
    print(f'Warning: iota_res={iota_res:.6f} is outside [{np.nanmin(iota):.6f}, {np.nanmax(iota):.6f}]. Use nearest surface.')
idx_res = int(np.nanargmin(np.abs(iota - iota_res)))
rho_res = float(rho[idx_res])
s_res = rho_res**2

internal_targets = [(m1, -n1), (m2, -n2)]
families = {n_internal % int(eq.NFP) for _, n_internal in internal_targets}
if len(families) != 1:
    raise ValueError(f'Target modes are not in one NFP={eq.NFP} family: {internal_targets}')
mode_family = families.pop()
level_names = {str(level).lower() for level in RUN_LEVELS}
n_pad_for_check = max([{'bare': 0, 'compact': 1, 'medium': 2, 'large': 3}.get(level, 0) for level in level_names] or [0])
m_pad_map = {'bare': 0, 'compact': 1, 'medium': 2, 'large': 3} if GAP_TYPE in {'TAE', 'MAE'} else {'bare': 0, 'compact': 1, 'medium': 2, 'large': 2}
m_pad_for_check = max([m_pad_map.get(level, 0) for level in level_names] or [0])
required_m_for_check = max(m1, m2) + m_pad_for_check
required_n_for_check = max(abs(n_internal) for _, n_internal in internal_targets) + n_pad_for_check * int(eq.NFP)
check_resolution(eq, required_m=required_m_for_check, required_n=required_n_for_check, axisymmetric=False)
case_tag = f'{eq_path.stem}_{GAP_TYPE.lower()}_n{n1}m{m1}_n{n2}m{m2}_{"snd" if SOUND_COUPLING else "alf"}'[:48]

profile_fit = globals().get('PROFILE_FIT')
if profile_fit is None:
    raise RuntimeError('请先运行第2格读取 NTP.mat。')

info = dict(
    equilibrium=str(eq_path), continuum_model=CONTINUUM_LABEL, gap_type=GAP_TYPE, NFP=int(eq.NFP),
    physical_modes_n_m=[list(pair) for pair in MODE_PAIR_NM], stellgap_modes_m_n=internal_targets,
    iota_axis=float(iota[0]), iota_edge=float(iota[-1]),
    iota_resonance=float(iota_res), q_resonance=float(q_res), rho_resonance=rho_res,
    density_m3=float(profile_fit['ni0_m3'] if SOUND_COUPLING else profile_fit['ni0_m3_s']),
    ion_profile=4 if SOUND_COUPLING else 1,
    ion_mass_in_protons=float(ION_MASS_IN_PROTONS),
)
if SOUND_COUPLING:
    info['sound_parameters'] = dict(
        gamma=float(SOUND_GAMMA),
        electron_temperature_ev=float(profile_fit['te0_ev']),
        te_profile=2,
        slow_sound=bool(SLOW_SOUND_APPROX),
    )
pprint(info)

In [ ]:
################## 5. 运行 STELLGAP ##################

wout = WORK_DIR / f'wout_{case_tag}.nc'
VMECIO.save(eq, wout, surfs=int(VMEC_SURFACES), M_nyq=int(VMEC_M_NYQ), N_nyq=int(VMEC_N_NYQ),
            M_grid=int(VMEC_M_GRID), N_grid=int(VMEC_N_GRID), verbose=1)

surfaces = list(range(2, int(VMEC_SURFACES)))
boozer_input = WORK_DIR / f'in_booz.{case_tag}'
boozer_lines = [f'{int(BOOZER_M)} {int(BOOZER_N)}', f"'{case_tag}'"]
boozer_lines += [' '.join(map(str, surfaces[i:i + 20])) for i in range(0, len(surfaces), 20)]
boozer_input.write_text('\n'.join(boozer_lines) + '\n', encoding='ascii')

run_command([TOOLS['booz_xform'], boozer_input.name], '01_booz_xform.log')
run_command([TOOLS['metric'], case_tag, '-itheta', str(int(METRIC_THETA)), '-izeta', str(int(METRIC_ZETA))], '02_xmetric.log')


def mode_rows(level):
    level = str(level).lower()
    target_by_n = {}
    for m, n_internal in internal_targets:
        target_by_n.setdefault(n_internal, []).append(m)
    if level == 'bare':
        return [(n_internal, min(ms), max(ms)) for n_internal, ms in sorted(target_by_n.items())]
    if level not in {'compact', 'medium', 'large'}:
        raise ValueError(f'Unknown run level: {level}')

    n_pad = {'compact': 1, 'medium': 2, 'large': 3}[level]
    m_pad = {'compact': 1, 'medium': 2, 'large': 3}[level] if GAP_TYPE in {'TAE', 'MAE'} else {'compact': 1, 'medium': 2, 'large': 2}[level]
    n_values = sorted(target_by_n)
    n_min = min(n_values) - n_pad * int(eq.NFP)
    n_max = max(n_values) + n_pad * int(eq.NFP)
    return [(n_internal, max(0, min(m1, m2) - m_pad), max(m1, m2) + m_pad)
            for n_internal in range(n_min, n_max + 1, int(eq.NFP))]


def write_stellgap_inputs(rows):
    lines = [f'{int(eq.NFP)} {int(METRIC_THETA)} {int(METRIC_ZETA)} {int(mode_family)}', str(len(rows))]
    lines += [f'{int(n_internal)} {int(m_min)} {int(m_max)}' for n_internal, m_min, m_max in rows]
    (WORK_DIR / 'fourier.dat').write_text('\n'.join(lines) + '\n', encoding='ascii')
    write_plasma_input()
    return sum(int(m_max) - int(m_min) + 1 for _, m_min, m_max in rows)


mode_info = {}
post_files = {}
for level in RUN_LEVELS:
    level = str(level).lower()
    rows = mode_rows(level)
    mode_count = write_stellgap_inputs(rows)
    clear_stellgap_outputs()
    run_command([TOOLS['stgap'], str(int(VMEC_SURFACES) - 2), str(int(RADIAL_POINTS))], f'03_{STELLGAP_PROGRAM}_{level}.log')
    run_command([TOOLS['post']], f'04_{POST_PROGRAM}_{level}.log')
    post_files[level] = WORK_DIR / f'alfven_post_{level}'
    shutil.copy2(WORK_DIR / 'alfven_post', post_files[level])
    mode_info[level] = {'rows': rows, 'mode_count': mode_count}
    print(level, mode_count, 'Fourier modes')

print('Spectra ready.')

In [ ]:
################## 6. 画图 ##################

def gap_at_resonance(data, targets):
    s_values = np.unique(data[:, 0])
    s_sample = float(s_values[np.argmin(np.abs(s_values - s_res))])
    rows = data[np.isclose(data[:, 0], s_sample)]
    freqs = []
    for m, n_internal in targets:
        hit = rows[np.isclose(rows[:, 2], m) & np.isclose(rows[:, 3], n_internal)]
        if len(hit) == 0:
            labels = sorted({(int(row[2]), int(row[3])) for row in rows})
            raise RuntimeError(f'Cannot find target label {(m, n_internal)} at s={s_sample:.6f}. Available labels: {labels}')
        freqs.append(float(hit[0, 1]))
    lo, hi = sorted(freqs)
    return dict(sample_s=s_sample, sample_rho=float(np.sqrt(max(s_sample, 0.0))),
                lower_kHz=lo, upper_kHz=hi, center_kHz=0.5 * (lo + hi), width_kHz=hi - lo,
                contains_reference=None if REFERENCE_FREQUENCY_KHZ is None else bool(lo <= REFERENCE_FREQUENCY_KHZ <= hi))


def crop_curve(curve):
    curve_rho = np.sqrt(np.clip(curve[:, 0], 0.0, None))
    return curve[(curve_rho >= rho_plot_min) & (curve_rho <= rho_plot_max)]


spectra = {level: read_stellgap_post(path) for level, path in post_files.items()}
convergence = {level: gap_at_resonance(data, internal_targets) for level, data in spectra.items()}
final_level = str(RUN_LEVELS[-1]).lower()
data = spectra[final_level]
gap = convergence[final_level]

(target_m1, target_n1), (target_m2, target_n2) = internal_targets
curve1 = select_branch(data, target_m1, target_n1)
curve2 = select_branch(data, target_m2, target_n2)
if len(curve1) == 0 or len(curve2) == 0:
    raise RuntimeError('Target branches were not found in final STELLGAP output.')

rho_plot_min, rho_plot_max = sorted(map(float, PLOT_RHO_RANGE))
if not (0.0 <= rho_plot_min < rho_plot_max <= 1.0):
    raise ValueError('PLOT_RHO_RANGE must satisfy 0 <= rho_min < rho_max <= 1')
rho_all = np.sqrt(np.clip(data[:, 0], 0.0, None))
if FREQUENCY_RANGE_KHZ is None:
    f_min = max(0.0, min(curve1[:, 1].min(), curve2[:, 1].min()) - 120.0)
    f_max = max(curve1[:, 1].max(), curve2[:, 1].max()) + 120.0
else:
    f_min, f_max = sorted(map(float, FREQUENCY_RANGE_KHZ))
    if f_min < 0.0 or f_max <= f_min:
        raise ValueError('FREQUENCY_RANGE_KHZ must be None or satisfy 0 <= f_min < f_max')
mask = (data[:, 1] >= f_min) & (data[:, 1] <= f_max) & (rho_all >= rho_plot_min) & (rho_all <= rho_plot_max)
curve1_plot, curve2_plot = crop_curve(curve1), crop_curve(curve2)

fig, (ax, zoom) = plt.subplots(1, 2, figsize=(16, 6.2))
fig.subplots_adjust(left=0.07, right=0.985, bottom=0.14, top=0.84, wspace=0.12)
fig.suptitle(f'{GAP_TYPE} {CONTINUUM_LABEL} continuum from DESC equilibrium {eq_path.stem}', fontsize=18, fontweight='bold', y=0.96)

ax.scatter(rho_all[mask], data[mask, 1], s=5, c='0.70', alpha=0.30, edgecolors='none', label=f'{mode_info[final_level]["mode_count"]}-mode truncation')
ax.scatter(np.sqrt(curve1_plot[:, 0]), curve1_plot[:, 1], s=13, c='#d62728', edgecolors='none', label=f'target m={m1}, n={n1}')
ax.scatter(np.sqrt(curve2_plot[:, 0]), curve2_plot[:, 1], s=13, c='#1f77b4', edgecolors='none', label=f'target m={m2}, n={n2}')
ax.axvline(rho_res, color='0.25', ls=':', lw=1.5, label=fr'resonance $q={q_res:.4f}$')
ax.set(xlim=(rho_plot_min, rho_plot_max), ylim=(f_min, f_max), xlabel=r'$\rho=\sqrt{s}$', ylabel='frequency [kHz]', title='Continuum and sidebands')
ax.grid(alpha=0.25)
ax.legend(fontsize=9.5)

for curve, color, label in [(curve1_plot, '#d62728', f'target m={m1}, n={n1}'), (curve2_plot, '#1f77b4', f'target m={m2}, n={n2}')]:
    zoom.scatter(np.sqrt(curve[:, 0]), curve[:, 1], s=15, c=color, edgecolors='none', label=label)

if 'bare' in spectra:
    for k, (m, n_internal) in enumerate(internal_targets):
        bare_curve = select_branch(spectra['bare'], m, n_internal)
        bare_curve = crop_curve(bare_curve) if len(bare_curve) else bare_curve
        if len(bare_curve):
            zoom.plot(np.sqrt(bare_curve[:, 0]), bare_curve[:, 1], color='0.45', ls='-.', lw=1.2, label='bare target truncation' if k == 0 else None)

zoom.axvline(rho_res, color='0.25', ls=':', lw=1.5, label=fr'resonance $\rho={rho_res:.3f}$')
if REFERENCE_FREQUENCY_KHZ is not None:
    zoom.axhline(float(REFERENCE_FREQUENCY_KHZ), color='black', ls='--', lw=1.2, label=f'{float(REFERENCE_FREQUENCY_KHZ):.1f} kHz reference')

pad = max(20.0, 0.35 * max(gap['width_kHz'], 1.0))
zoom_ymin, zoom_ymax = gap['lower_kHz'] - pad, gap['upper_kHz'] + pad
if FREQUENCY_RANGE_KHZ is not None:
    zoom_ymin, zoom_ymax = f_min, f_max
gap_visible = zoom_ymin <= gap['lower_kHz'] and gap['upper_kHz'] <= zoom_ymax
zoom.set(
    xlim=(max(rho_plot_min, gap['sample_rho'] - 0.16), min(rho_plot_max, gap['sample_rho'] + 0.16)),
    ylim=(zoom_ymin, zoom_ymax),
    xlabel=r'$\rho=\sqrt{s}$',
    title='Target gap zoom',
)
zoom.set_ylabel('frequency [kHz]')
zoom.grid(alpha=0.25)
if gap_visible:
    zoom.annotate('', xy=(gap['sample_rho'], gap['upper_kHz']), xytext=(gap['sample_rho'], gap['lower_kHz']),
                  arrowprops=dict(arrowstyle='<->', color='black', lw=1.6))
    zoom.text(min(rho_plot_max, gap['sample_rho'] + 0.03), gap['center_kHz'],
              f'gap width={gap["width_kHz"]:.1f} kHz\ncenter={gap["center_kHz"]:.1f} kHz',
              va='center', fontsize=10.5)
elif FREQUENCY_RANGE_KHZ is not None:
    print('Warning: target gap is outside FREQUENCY_RANGE_KHZ; skip gap annotation.')
zoom.legend(fontsize=9.5)

print('Target gap:')
pprint(gap)
print('Convergence by truncation:')
pprint(convergence)
if SOUND_COUPLING and data.shape[1] >= 6:
    print('Sound-coupled alfven_post columns: s, frequency[kHz], dominant Alfven m/n, Alfven norm, sound norm.')
plt.show()
TEMP_DIR.cleanup()